# 🌟 ENTRENAMIENTO MISTRAL-7B CON LoRA

---

## 📊 Especificaciones

- **Modelo:** Mistral-7B-Instruct-v0.3 (7B parámetros)
- **Empresa:** Mistral AI (Francia) 🇫🇷
- **Técnica:** QLoRA (4-bit)
- **GPU:** T4 (15GB VRAM)
- **Tiempo:** 40-50 minutos
- **Calidad español:** ⭐⭐⭐⭐⭐ **MEJOR que Llama**

---

## ✅ VENTAJAS DE MISTRAL

- 🌟 **MEJOR español** que Llama-3
- ✅ **Usado por empresas** (Fortune 500)
- ✅ **Licencia Apache 2.0** (comercial libre)
- ✅ **Más rápido** que Llama-3-8B
- ✅ **Excelente documentación**
- ✅ **Actualizaciones constantes**

---

## 🏢 EMPRESAS QUE USAN MISTRAL

- Microsoft Azure
- Cloudflare
- Brave Browser
- Startups europeas
- Gobiernos europeos

---

## 📦 PASO 1: INSTALAR DEPENDENCIAS

In [ ]:
%%capture
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets sentencepiece einops
print("✅ Dependencias instaladas")

## 🎮 PASO 2: VERIFICAR GPU

In [ ]:
import torch
print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ GPU NO disponible")
print("=" * 70)

## 📤 PASO 3: SUBIR DATASET

In [ ]:
from google.colab import files
import json

print("📤 Sube dataset_pedagogico.json")
uploaded = files.upload()

if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n✅ Dataset: {len(data)} ejemplos")
else:
    print("❌ Error: No se encontró el archivo")

## 📊 PASO 4: ABRIR TENSORBOARD

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/mistral
print("📊 TensorBoard abierto (se actualiza cada 30s)")

## 🏋️ PASO 5: ENTRENAR MISTRAL-7B

**Tiempo:** 40-50 minutos

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

print("=" * 70)
print("🌟 ENTRENAMIENTO MISTRAL-7B CON QLoRA")
print("=" * 70)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# Cuantización 4-bit
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Training config
training_args = TrainingArguments(
    output_dir="./lora_model",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_dir="./logs/mistral",
    logging_steps=5,
    save_steps=100,
    warmup_steps=30,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    disable_tqdm=False,
    logging_first_step=True,
)

print(f"\n📦 Modelo: {MODEL_NAME}")
print(f"🔧 LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"⏱️  Tiempo estimado: 40-50 minutos")
print(f"🌟 Calidad español: EXCELENTE")

# Preparar dataset
print("\n📚 Preparando dataset...")

with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    """Formato Mistral (usa [INST] tags)"""
    text = f"""<s>[INST] Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.

{example['instruction']}

{example['input']} [/INST] {example['output']}</s>"""
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction)
print(f"   ✅ {len(dataset)} ejemplos preparados")

# Cargar modelo
print("\n🤖 Cargando Mistral-7B con cuantización 4-bit...")
print("   (Descargando ~14GB, puede tardar 3-5 minutos)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

print(f"   ✅ Modelo cargado en GPU (4-bit)")

model = prepare_model_for_kbit_training(model)

# Aplicar LoRA
print("\n🔧 Aplicando adaptadores LoRA...")
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n📊 ESTADÍSTICAS:")
print(f"   Total: {total_params:,} parámetros")
print(f"   Entrenables: {trainable_params:,} ({100*trainable_params/total_params:.4f}%)")

# Tokenizar
print("\n📝 Tokenizando dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"   ✅ Dataset tokenizado")

# Entrenar
print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)
print(f"⏱️  Tiempo estimado: 40-50 minutos")
print(f"📉 Loss esperado: 1.5 → 0.3-0.5")
print(f"📊 Mira TensorBoard arriba ↑")
print()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("🔥 Empezando entrenamiento...")
trainer.train()

print("\n" + "=" * 70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("=" * 70)

final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"\n📊 Loss final: {final_loss}")

if isinstance(final_loss, float):
    if final_loss < 0.3:
        print("✅ ¡EXCELENTE! Loss <0.3")
    elif final_loss < 0.5:
        print("✅ MUY BIEN! Loss <0.5")
    else:
        print("⚠️  Aceptable")

# Guardar
print("\n💾 Guardando adaptadores...")
model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")
print(f"   ✅ Guardado en: ./lora_adapters")

# Probar
print("\n🧪 PROBANDO MODELO...")
print("=" * 70)

model.eval()

test_prompt = """<s>[INST] Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.

Explica qué es una derivada

Necesito entender el concepto de derivada [/INST] """

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("📝 RESPUESTA:")
print("-" * 70)
print(response)
print("-" * 70)

print("\n✅ PROCESO COMPLETADO")

## 📥 PASO 6: DESCARGAR ADAPTADORES

In [ ]:
import shutil
from google.colab import files

print("📦 Comprimiendo adaptadores...")
shutil.make_archive('lora_adapters_mistral_7b', 'zip', './lora_adapters')

print("✅ Adaptadores comprimidos")
print("\n📥 Descargando...")

files.download('lora_adapters_mistral_7b.zip')

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime lora_adapters_mistral_7b.zip")
print("   2. Renombra a 'lora_adapters'")
print("   3. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   4. Actualiza .env:")
print("      HUGGINGFACE_MODEL=mistralai/Mistral-7B-Instruct-v0.3")
print("   5. Reinicia el backend")
print("\n🌟 ¡Mistral-7B listo! (Mejor español que Llama)")